# Wan 2.2 image-to-video on Kaggle (free T4)

Turn **your images + prompts** into short video clips with the open **Wan 2.2 14B image-to-video** model and the
**LightX2V 4-step** speed-up, running in ComfyUI on Kaggle's free GPU.

**Before you run**
1. Notebook settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).
2. Add your inputs as a Kaggle Dataset (**+ Add Input**), or upload them to `/kaggle/working/jobs/`:
   - `shot01.png` + `shot01.txt` (the prompt), `shot02.jpg` + `shot02.txt`, ...  **or**
   - images + a `jobs.csv` with columns `image,prompt` (optional `seconds,seed,name`).
   - The image is the **first frame**: frame it exactly like the shot you want (same angle, lens, framing), 9:16.
3. **Run All.** With no inputs it makes one short test clip so you can time the GPU.

**Two modes** (set `MODE` in the config cell)
- `batch`: render every job, save MP4s to `/kaggle/working/outputs/` (download from the Output tab, or "Save Version").
- `tunnel`: start ComfyUI and print a public URL for **OpenMontage** on your laptop (`COMFYUI_VIDEO_SERVER_URL`).

**Limits to know:** about 30 GPU hours a week, 12 h per session. One Kaggle account per person (Kaggle's terms).
A T4 is old: expect minutes per clip, not seconds. The timing test tells you the real number.

In [ ]:
# ===================== CONFIG =====================
MODE      = "batch"      # "batch" or "tunnel"
MODEL     = "gguf_q4"    # "gguf_q4" (safest on Kaggle RAM), "gguf_q5" (a bit better), "fp8" (what OpenMontage's own workflow expects)
WIDTH, HEIGHT = 480, 832 # 9:16, multiples of 16. 576x1024 looks better but is ~1.6x slower
SECONDS   = 5            # Wan runs at 16 fps; 5 s = 81 frames (the model's sweet spot)
SEED      = -1           # -1 = random per clip
SHIFT     = 5.0          # model sampling shift (5 is the usual for the 4-step LoRA)
UPSCALE_TO_1080 = True   # also save a 1080x1920 copy (ffmpeg lanczos)
NEGATIVE  = ("blurry, low quality, jpeg artifacts, static frame, deformed hands, extra fingers, "
             "warped face, flicker, subtitles, text, watermark")
INPUT_DIR = None         # None = auto: first folder under /kaggle/input with images, else /kaggle/working/jobs
# ==================================================
import os, pathlib
WORK   = pathlib.Path("/kaggle/working")
OUT    = WORK / "outputs";  OUT.mkdir(parents=True, exist_ok=True)
COMFY  = pathlib.Path("/tmp/ComfyUI")          # big scratch disk, not the 20 GB /kaggle/working
FRAMES = 4 * round((SECONDS * 16) / 4) + 1      # Wan needs 4n+1 frames
assert WIDTH % 16 == 0 and HEIGHT % 16 == 0, "width/height must be multiples of 16"
print(f"mode={MODE} model={MODEL} size={WIDTH}x{HEIGHT} frames={FRAMES} (~{FRAMES/16:.1f} s)")

In [ ]:
# Check the machine
import subprocess, shutil, psutil
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(f"RAM {psutil.virtual_memory().total/2**30:.1f} GB   /tmp free {shutil.disk_usage('/tmp').free/2**30:.0f} GB   "
      f"/kaggle/working free {shutil.disk_usage('/kaggle/working').free/2**30:.0f} GB")
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Install ComfyUI (+ the GGUF loader). Keeps Kaggle's own torch so nothing big is re-downloaded.
import subprocess, sys
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(cmd)
    return r.stdout
if not COMFY.exists():
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}")
    sh(f"git clone --depth 1 https://github.com/city96/ComfyUI-GGUF {COMFY}/custom_nodes/ComfyUI-GGUF")
req = [l for l in open(COMFY/"requirements.txt").read().splitlines()
       if l.strip() and not l.startswith("#") and not l.split("=")[0].split(">")[0].split("<")[0].strip() in ("torch","torchvision","torchaudio")]
open("/tmp/req.txt","w").write("\n".join(req))
sh(f"{sys.executable} -m pip install -q -r /tmp/req.txt -r {COMFY}/custom_nodes/ComfyUI-GGUF/requirements.txt av huggingface_hub")
print("ComfyUI ready")

In [ ]:
# Use your saved model dataset if it's attached (any input folder named wan22-i2v-models).
# The files are COPIED to the local disk, not linked: reading 25+ GB of weights straight from /kaggle/input
# is slow and can stall a clip for a long time. The copy takes a few minutes once per session.
import glob, shutil, time
saved = [pathlib.Path(p) for p in glob.glob("/kaggle/input/**/wan22-i2v-models", recursive=True) if os.path.isdir(p)]
if saved:
    src = saved[0]; M = COMFY / "models"; t0 = time.time()
    for f in sorted(src.iterdir()):
        n = f.name
        if not n.endswith((".gguf", ".safetensors")): continue
        folder = ("loras" if "lora" in n else "text_encoders" if "umt5" in n else "vae" if "vae" in n
                  else "unet" if n.endswith(".gguf") else "diffusion_models")
        dest = M / folder / n; dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.is_symlink(): dest.unlink()
        if dest.exists() and dest.stat().st_size == f.stat().st_size: print("have", n); continue
        print(f"copying {n} ({f.stat().st_size/2**30:.1f} GB) -> {folder}/"); shutil.copyfile(f, dest)
    print(f"models from {src} ready in {(time.time()-t0)/60:.1f} min")
else:
    print("no saved model dataset attached: the next cell downloads from Hugging Face")

In [ ]:
# Download the model files from Hugging Face (first run: ~25-35 GB, 10-20 min).
# Tip: after the first run, save /tmp/ComfyUI/models as a Kaggle Dataset and copy from it next time.
import os
from huggingface_hub import hf_hub_download, list_repo_files
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # optional, for faster downloads
except Exception:
    pass

R22, R21 = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "Comfy-Org/Wan_2.1_ComfyUI_repackaged"
M = COMFY / "models"
def get(repo, path, folder):
    dest = M / folder / pathlib.Path(path).name
    if dest.exists(): print("have", dest.name); return dest.name
    print("downloading", repo, path)
    f = hf_hub_download(repo, path, local_dir="/tmp/hf")
    dest.parent.mkdir(parents=True, exist_ok=True); os.replace(f, dest)
    return dest.name

TEXT_ENC = get(R22, "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders")
VAE      = get(R21, "split_files/vae/wan_2.1_vae.safetensors", "vae")
LORA_HI  = get(R22, "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "loras")
LORA_LO  = get(R22, "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "loras")

if MODEL == "fp8":
    UNET_HI = get(R22, "split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", "diffusion_models")
    UNET_LO = get(R22, "split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors", "diffusion_models")
else:
    GGUF_REPO = "QuantStack/Wan2.2-I2V-A14B-GGUF"
    quant = {"gguf_q4": "Q4_K_M", "gguf_q5": "Q5_K_M"}[MODEL]
    files = [f for f in list_repo_files(GGUF_REPO) if f.endswith(".gguf") and quant in f]
    hi = [f for f in files if "high" in f.lower()]; lo = [f for f in files if "low" in f.lower()]
    assert hi and lo, f"no {quant} files found in {GGUF_REPO}: {files}"
    UNET_HI = get(GGUF_REPO, hi[0], "unet"); UNET_LO = get(GGUF_REPO, lo[0], "unet")
print("models:", UNET_HI, UNET_LO, TEXT_ENC, VAE, LORA_HI, LORA_LO)

In [ ]:
# Start ComfyUI in the background on GPU 0 and wait until it answers. start_comfy() is reused if it ever dies.
import subprocess, time, urllib.request, urllib.error, json
PORT = 8188
comfy_proc = None
def start_comfy():
    global comfy_proc
    log = open("/tmp/comfy.log", "a")   # append, so a crash's reason survives a restart
    log.write(f"\n===== ComfyUI start {time.strftime('%H:%M:%S')} =====\n"); log.flush()
    comfy_proc = subprocess.Popen([sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT),
                                   "--cuda-device", "0", "--preview-method", "none"],
                                  cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/system_stats", timeout=2); return
        except Exception:
            if comfy_proc.poll() is not None: break
            time.sleep(2)
    print(open("/tmp/comfy.log", errors="ignore").read()[-4000:]); raise RuntimeError("ComfyUI did not start (log above)")

def why_dead():
    code = comfy_proc.poll()
    hint = {-9: "killed: the machine ran out of RAM", 137: "killed: the machine ran out of RAM"}.get(code, "see the log")
    return f"ComfyUI exited with code {code} ({hint}). Last log lines:\n" + open("/tmp/comfy.log", errors="ignore").read()[-2500:]

def ensure_comfy():
    if comfy_proc is None or comfy_proc.poll() is not None:
        if comfy_proc is not None: print(why_dead())
        print("starting ComfyUI..."); start_comfy()

ensure_comfy()
print("ComfyUI is up on port", PORT)

In [ ]:
# The Wan 2.2 image-to-video graph (standard 2-pass high/low-noise, 4 steps) and a tiny client.
import uuid, random, glob

def build_workflow(image_name, prompt, frames, seed, prefix):
    if MODEL == "fp8":
        loader = lambda n: {"class_type": "UNETLoader", "inputs": {"unet_name": n, "weight_dtype": "default"}}
    else:
        loader = lambda n: {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": n}}
    ks = lambda model, add, s0, s1, lat, left: {"class_type": "KSamplerAdvanced", "inputs": {
        "model": model, "add_noise": add, "noise_seed": seed, "steps": 4, "cfg": 1.0,
        "sampler_name": "euler", "scheduler": "simple", "start_at_step": s0, "end_at_step": s1,
        "return_with_leftover_noise": left, "positive": ["img2vid", 0], "negative": ["img2vid", 1],
        "latent_image": lat}}
    return {
        "clip":     {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENC, "type": "wan", "device": "default"}},
        "pos":      {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["clip", 0]}},
        "neg":      {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVE, "clip": ["clip", 0]}},
        "vae":      {"class_type": "VAELoader", "inputs": {"vae_name": VAE}},
        "image":    {"class_type": "LoadImage", "inputs": {"image": image_name}},
        "img2vid":  {"class_type": "WanImageToVideo", "inputs": {"positive": ["pos", 0], "negative": ["neg", 0],
                     "vae": ["vae", 0], "start_image": ["image", 0], "width": WIDTH, "height": HEIGHT,
                     "length": frames, "batch_size": 1}},
        "unet_hi":  loader(UNET_HI),
        "unet_lo":  loader(UNET_LO),
        "lora_hi":  {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["unet_hi", 0], "lora_name": LORA_HI, "strength_model": 1.0}},
        "lora_lo":  {"class_type": "LoraLoaderModelOnly", "inputs": {"model": ["unet_lo", 0], "lora_name": LORA_LO, "strength_model": 1.0}},
        "shift_hi": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["lora_hi", 0], "shift": SHIFT}},
        "shift_lo": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["lora_lo", 0], "shift": SHIFT}},
        "pass1":    ks(["shift_hi", 0], "enable", 0, 2, ["img2vid", 2], "enable"),
        "pass2":    ks(["shift_lo", 0], "disable", 2, 4, ["pass1", 0], "disable"),
        "decode":   {"class_type": "VAEDecode", "inputs": {"samples": ["pass2", 0], "vae": ["vae", 0]}},
        "video":    {"class_type": "CreateVideo", "inputs": {"images": ["decode", 0], "fps": 16}},
        "save":     {"class_type": "SaveVideo", "inputs": {"video": ["video", 0], "filename_prefix": prefix,
                     "format": "auto", "codec": "auto"}},
    }

def api(path, data=None):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}",
                                 data=json.dumps(data).encode() if data is not None else None,
                                 headers={"Content-Type": "application/json"})
    try:
        return json.loads(urllib.request.urlopen(req, timeout=60).read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(e.read().decode()[:3000])

import re, psutil
from IPython.display import display

def status_line():
    txt = open("/tmp/comfy.log", errors="ignore").read()[-6000:]
    parts = [p.strip() for p in re.split(r"[\r\n]", txt) if p.strip()]
    bar = next((p for p in reversed(parts) if "%|" in p), "loading models / encoding...")
    gpu = subprocess.run(["nvidia-smi", "-i", "0", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    ram = psutil.virtual_memory()
    return f"{bar[-80:]} | GPU {gpu} | RAM {ram.used/2**30:.1f}/{ram.total/2**30:.0f} GB"

def finish(src, name):
    out = OUT / f"{name}.mp4"; shutil.copy(src, out)
    if UPSCALE_TO_1080:
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(out), "-vf",
                        "scale=1080:1920:flags=lanczos,setsar=1", "-c:v", "libx264", "-crf", "18",
                        "-pix_fmt", "yuv420p", "-c:a", "copy", str(OUT / f"{name}_1080.mp4")], check=False)
    return out

def render(image_path, prompt, name, seconds=SECONDS, seed=SEED):
    ensure_comfy()
    done = sorted(glob.glob(str(COMFY / "output" / f"kaggle/{name}_*")), key=os.path.getmtime)
    if done:   # finished in the background earlier (e.g. after you stopped the cell)
        print(f"  {name}: picked up the finished file -> {finish(done[-1], name)}"); return 0.0, -1
    t0 = time.time(); line = display("", display_id=True)
    while api("/queue").get("queue_running"):   # an earlier job is still running: let it finish first
        line.update(f"  waiting for an earlier job, {(time.time()-t0)/60:.1f} min | {status_line()}"); time.sleep(10)
    done = sorted(glob.glob(str(COMFY / "output" / f"kaggle/{name}_*")), key=os.path.getmtime)
    if done:
        print(f"  {name}: the earlier job made it -> {finish(done[-1], name)}"); return time.time() - t0, -1
    frames = 4 * round((float(seconds) * 16) / 4) + 1
    seed = random.randint(0, 2**31 - 1) if int(seed) < 0 else int(seed)
    inp = COMFY / "input"; inp.mkdir(exist_ok=True)
    img_name = f"{name}{pathlib.Path(image_path).suffix}"
    shutil.copy(image_path, inp / img_name)
    prefix = f"kaggle/{name}_{seed}"
    t0 = time.time()
    pid = api("/prompt", {"prompt": build_workflow(img_name, prompt, frames, seed, prefix),
                          "client_id": str(uuid.uuid4())})["prompt_id"]
    while True:
        time.sleep(10)
        if comfy_proc.poll() is not None: raise RuntimeError(why_dead())
        line.update(f"  {name}: {(time.time()-t0)/60:.1f} min | {status_line()}")
        h = api(f"/history/{pid}").get(pid)
        if h:
            st = h.get("status", {})
            if st.get("status_str") == "error":
                raise RuntimeError(json.dumps(st.get("messages", []))[-3000:])
            if st.get("completed"): break
    found = sorted(glob.glob(str(COMFY / "output" / f"{prefix}*")), key=os.path.getmtime)
    if not found: raise RuntimeError("finished but no video file found")
    out = finish(found[-1], name); secs = time.time() - t0
    print(f"  done {name}: {frames} frames in {secs/60:.1f} min (seed {seed}) -> {out}")
    return secs, seed
print("ready")

In [ ]:
# Find the jobs: <name>.png + <name>.txt pairs, or a jobs.csv (image,prompt[,seconds,seed,name]).
import csv, re
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp"}
def find_input_dir():
    if INPUT_DIR: return pathlib.Path(INPUT_DIR)
    for d in [WORK / "jobs", *sorted(pathlib.Path("/kaggle/input").glob("**/"))]:
        if d.is_dir() and any(p.suffix.lower() in IMG_EXT for p in d.iterdir() if p.is_file()):
            return d
    return None

def load_jobs(d):
    jobs = []
    if (d / "jobs.csv").exists():
        for row in csv.DictReader(open(d / "jobs.csv", encoding="utf-8")):
            img = d / row["image"].strip()
            jobs.append({"image": img, "prompt": row["prompt"].strip(),
                         "seconds": row.get("seconds") or SECONDS, "seed": row.get("seed") or SEED,
                         "name": (row.get("name") or img.stem).strip()})
    else:
        # match prompts by name, ignoring case and leading zeros: shot1.png <-> shot01.txt
        key = lambda stem: re.sub(r"\d+", lambda m: str(int(m.group())), stem.lower())
        txts = {key(t.stem): t for t in d.glob("*.txt")}
        for img in sorted((p for p in d.iterdir() if p.suffix.lower() in IMG_EXT), key=lambda p: re.sub(r"\d+", lambda m: m.group().zfill(6), p.stem.lower())):
            txt = img.with_suffix(".txt")
            if not txt.exists(): txt = txts.get(key(img.stem))
            if not txt: print("skip (no .txt prompt):", img.name); continue
            jobs.append({"image": img, "prompt": txt.read_text(encoding="utf-8").strip(),
                         "seconds": SECONDS, "seed": SEED, "name": img.stem})
    return jobs

JOB_DIR = find_input_dir()
JOBS = load_jobs(JOB_DIR) if JOB_DIR else []
print(f"input folder: {JOB_DIR}   jobs: {len(JOBS)}")
for j in JOBS: print(" -", j["name"], "|", j["prompt"][:90])

In [ ]:
# Timing test (only when there are no jobs): a 2 s clip from a generated test image.
if MODE == "batch" and not JOBS:
    from PIL import Image, ImageDraw
    im = Image.new("RGB", (WIDTH, HEIGHT), (20, 24, 40)); d = ImageDraw.Draw(im)
    for y in range(HEIGHT): d.line([(0, y), (WIDTH, y)], fill=(30 + y // 8, 40 + y // 10, 90))
    d.ellipse([WIDTH*0.3, HEIGHT*0.15, WIDTH*0.7, HEIGHT*0.38], fill=(250, 210, 120))
    d.rectangle([0, HEIGHT*0.7, WIDTH, HEIGHT], fill=(15, 60, 70))
    im.save("/tmp/test.png")
    secs, _ = render("/tmp/test.png", "A slow push-in on a glowing sun setting over a calm sea, gentle waves, "
                     "warm light rippling on the water, clouds drifting.", "timing_test", seconds=2, seed=1)
    per5 = secs * (81 / 33)
    print(f"\nTiming: {secs/60:.1f} min for 2 s (includes loading the models the first time).")
    print(f"Rough guess for a 5 s clip after warm-up: {per5/60*0.7:.0f}-{per5/60:.0f} min. "
          "Add your jobs and run the batch cell.")

In [ ]:
# Batch: render every job, skip ones already in outputs/, log timings.
if MODE == "batch" and JOBS:
    log_path = OUT / "timings.csv"; new = not log_path.exists()
    with open(log_path, "a", newline="") as f:
        w = csv.writer(f)
        if new: w.writerow(["name", "seconds_of_video", "render_minutes", "seed", "model", "size"])
        for i, j in enumerate(JOBS, 1):
            if (OUT / f"{j['name']}.mp4").exists(): print("already done:", j["name"]); continue
            print(f"[{i}/{len(JOBS)}] {j['name']}")
            for attempt in (1, 2):   # one retry, after restarting ComfyUI if it died
                try:
                    secs, seed = render(j["image"], j["prompt"], j["name"], j["seconds"], j["seed"])
                    w.writerow([j["name"], j["seconds"], f"{secs/60:.1f}", seed, MODEL, f"{WIDTH}x{HEIGHT}"]); f.flush()
                    break
                except Exception as e:
                    print(f"  FAILED (attempt {attempt}):", str(e)[:2500])
    print("\nOutputs:", sorted(p.name for p in OUT.glob("*.mp4")))

In [ ]:
# Contact sheet of the results (one row per clip), so you can judge them without downloading.
from IPython.display import Image as IPImage, display
for mp4 in sorted(OUT.glob("*.mp4")):
    if mp4.stem.endswith("_1080"): continue
    sheet = OUT / f"{mp4.stem}_sheet.jpg"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(mp4), "-vf", "fps=2,scale=160:-2,tile=6x1",
                    "-frames:v", "1", str(sheet)])
    if sheet.exists(): print(mp4.name); display(IPImage(filename=str(sheet)))

In [ ]:
# Tunnel mode: expose ComfyUI to your laptop for OpenMontage. Keep this cell running.
# Note: OpenMontage's bundled Wan workflow asks for the fp8 model files, so set MODEL = "fp8" for this mode.
# fp8 needs more RAM than GGUF; if ComfyUI crashes, use batch mode and give the clips to OpenMontage as assets.
if MODE == "tunnel":
    import re
    if not pathlib.Path("/tmp/cloudflared").exists():
        sh("wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared")
    tlog = open("/tmp/tunnel.log", "w")
    tun = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                           stdout=tlog, stderr=subprocess.STDOUT)
    url = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/tunnel.log").read())
        if m: url = m.group(0); break
        time.sleep(2)
    if not url: print(open("/tmp/tunnel.log").read()[-2000:]); raise RuntimeError("no tunnel URL")
    print("ComfyUI URL:", url)
    print("\nOn your laptop, in OpenMontage's .env:\n  COMFYUI_VIDEO_SERVER_URL=" + url)
    print("\nAnyone with this URL can use your ComfyUI while it runs: don't share it. Stop this cell to close it.")
    try:
        while comfy_proc.poll() is None and tun.poll() is None: time.sleep(30)
    finally:
        tun.terminate()

## If something goes wrong
- **"Connection refused"**: ComfyUI is not running (it crashed, or the kernel was restarted, which stops it too).
  The batch cell now restarts it and prints why it died. Exit code -9 means it ran out of RAM.
- **A clip shows no progress for a long time**: watch the live line. "loading models" for many minutes means the
  model files are being read slowly; attach them as a dataset so they're copied to local disk (the saved-models cell).
- **ComfyUI crashed / "Killed"**: out of system RAM. Use `MODEL = "gguf_q4"`, and don't run anything else in the notebook.
- **CUDA out of memory**: lower the size (`WIDTH, HEIGHT = 480, 832` is the safe default) or `SECONDS = 3`.
- **Black or noisy video**: T4 has no bf16, so ComfyUI runs in fp16. Add `"--fp32-vae"` to the ComfyUI start command and restart.
- **"no Q4_K_M files found"**: the GGUF repo renamed its files. The error lists what's there; adjust `quant` or the repo name.
- **Slow downloads**: add a free Hugging Face token as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets).
- **Don't re-download every session**: after a run, create a Kaggle Dataset from `/tmp/ComfyUI/models`
  (or save the notebook output) and add it as an input; then copy it into `/tmp/ComfyUI/models` before the download cell.

## Prompt tips for Wan image-to-video
- The image already sets the look: the prompt should describe **motion**: what moves, how fast, and the camera move.
- One continuous shot per clip. Say the action in time order ("slowly rises... then... at the end...").
- Keep the camera simple: "locked camera", "slow push-in", "slow pan left". Fast moves warp on a 4-step model.